# CAMeL-BERT Binary Classification Fine-tuning
## Piste 1 : Boundary Detection sans BIO Tagging

**Objectif** : Fine-tune CAMeL-BERT pour détecter les limites de segments (isnad vs matn/poetry/prose)

**Approche** :
- Classification binaire par segment : label=1 si type=='isnad', label=0 sinon
- Chaque akhbar contient plusieurs segments
- Fine-tune depuis Google Drive directement

**Timeline** :
1. 🔗 Montage Drive + chargement dépendances
2. 📊 Préparation des données (tokenization)
3. 🤖 Fine-tuning CAMeL-BERT
4. 📈 Évaluation (accuracy, F1, confusion matrix)
5. 💾 Sauvegarde modèle + résultats sur Drive

---
## Step 1 : Montage Google Drive + Installation des dépendances

In [2]:
# Montage du Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/khabar-segmentation')
print(f"Working directory: {os.getcwd()}")
print(f"Files in current dir: {os.listdir('.')[:10]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/khabar-segmentation
Files in current dir: ['openiti_targeted', 'camelbert_binary_classification_inference.ipynb', 'data', 'camelbert_binary_classification_finetuning.ipynb']


In [3]:
# Installation des dépendances
!pip install transformers torch datasets scikit-learn tqdm -q
print("[OK] Dependencies installed")

[OK] Dependencies installed


In [19]:
# Imports
import json
import numpy as np
import torch
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

print("[OK] All imports successful")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

[OK] All imports successful
GPU available: True
GPU: Tesla T4


---
## Step 2 : Chargement et Préparation des Données

In [20]:
# Charger le fichier de données préparées
data_file = Path('data/processed/binary_classification_dataset/binary_classification_examples.jsonl')

if not data_file.exists():
    print(f"ERROR: {data_file} not found!")
    print(f"Available files in data/processed/:")
    print(list(Path('data/processed/').glob('*')))
else:
    # Charger les exemples
    examples = []
    with open(data_file, encoding='utf-8') as f:
        for line in f:
            examples.append(json.loads(line))

    print(f"[OK] Loaded {len(examples)} examples")
    print(f"\nExample 1:")
    ex = examples[0]
    print(f"  akhbar_id: {ex['akhbar_id']}")
    print(f"  has_isnad: {ex['has_isnad']}")
    print(f"  segments: {ex['segment_types']}")
    print(f"  num_segments: {len(ex['segments'])}")

[OK] Loaded 613 examples

Example 1:
  akhbar_id: 1
  has_isnad: False
  segments: ['prose', 'prose', 'prose', 'prose']
  num_segments: 4


In [21]:
# Tokenizer CAMeL-BERT
model_name = "CAMeL-Lab/bert-base-arabic-camelbert-ca"
print(f"[INFO] Loading tokenizer: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"[OK] Tokenizer loaded. Vocab size: {len(tokenizer)}")

[INFO] Loading tokenizer: CAMeL-Lab/bert-base-arabic-camelbert-ca
[OK] Tokenizer loaded. Vocab size: 30000


In [22]:
def tokenize_akhbar(akhbar_data: dict, tokenizer, max_length: int = 512) -> dict:
    """
    Tokenize un akhbar et créer les labels.

    Stratégie :
    - Concaténer tous les segments avec séparateur [SEP]
    - Assigner label=1 aux tokens du segment 'isnad', label=0 sinon
    - Truncate si > max_length
    """
    segments = akhbar_data['segments']

    # Construire le texte concaténé et tracer les types
    segment_texts = []
    segment_types = []

    for seg in segments:
        segment_texts.append(seg['text'])
        segment_types.append(seg['type'])

    # Joindre avec séparateur
    full_text = ' [SEP] '.join(segment_texts)

    # Tokenizer
    encoded = tokenizer(
        full_text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors=None,
        add_special_tokens=True
    )

    # Créer les labels
    # Approche simple : chercher si le token appartient à un segment 'isnad'
    labels = []
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'])

    # Reconstruire le mapping token -> segment_type
    # On va tokenizer chaque segment séparément, puis fusionner
    segment_token_labels = []

    for seg_text, seg_type in zip(segment_texts, segment_types):
        seg_encoded = tokenizer(
            seg_text,
            add_special_tokens=False,
            return_tensors=None
        )
        seg_len = len(seg_encoded['input_ids'])
        seg_label = 1 if seg_type == 'isnad' else 0
        segment_token_labels.extend([seg_label] * seg_len)

    # Créer les labels pour le texte complet
    # [CLS] + tokens du texte + [SEP]
    full_labels = [0] + segment_token_labels  # [CLS] = 0

    # Truncate aux labels et pad
    if len(full_labels) < max_length:
        full_labels.extend([0] * (max_length - len(full_labels)))  # padding = 0
    else:
        full_labels = full_labels[:max_length]

    return {
        'input_ids': encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
        'labels': full_labels,
        'akhbar_id': akhbar_data['akhbar_id'],
        'segment_types': akhbar_data['segment_types']
    }

print("[OK] tokenize_akhbar function defined")

[OK] tokenize_akhbar function defined


In [23]:
# Tokenizer tous les exemples
print("[INFO] Tokenizing all examples...")

tokenized_examples = []
skipped = 0

for i, ex in enumerate(tqdm(examples, desc="Tokenizing")):
    try:
        tokenized = tokenize_akhbar(ex, tokenizer)
        tokenized_examples.append(tokenized)
    except Exception as e:
        print(f"[SKIP] Example {i}: {e}")
        skipped += 1

print(f"[OK] Tokenized {len(tokenized_examples)} examples (skipped {skipped})")

# Afficher stats des labels
label_stats = defaultdict(int)
for ex in tokenized_examples:
    for label in ex['labels']:
        label_stats[label] += 1

print(f"\n[LABEL STATS]")
for label, count in sorted(label_stats.items()):
    pct = 100 * count / sum(label_stats.values())
    print(f"  Label {label} (boundary): {count} ({pct:.1f}%)")

[INFO] Tokenizing all examples...


Tokenizing: 100%|██████████| 613/613 [00:01<00:00, 536.64it/s]


[OK] Tokenized 613 examples (skipped 0)

[LABEL STATS]
  Label 0 (boundary): 297886 (94.9%)
  Label 1 (boundary): 15970 (5.1%)


In [24]:
# Créer train/val/test split
from sklearn.model_selection import train_test_split

# 70% train, 15% val, 15% test
indices = list(range(len(tokenized_examples)))
train_indices, temp_indices = train_test_split(indices, test_size=0.30, random_state=42)
val_indices, test_indices = train_test_split(temp_indices, test_size=0.50, random_state=42)

train_data = [tokenized_examples[i] for i in train_indices]
val_data = [tokenized_examples[i] for i in val_indices]
test_data = [tokenized_examples[i] for i in test_indices]

print(f"[SPLIT]")
print(f"  Train: {len(train_data)} ({100*len(train_data)/len(tokenized_examples):.1f}%)")
print(f"  Val:   {len(val_data)} ({100*len(val_data)/len(tokenized_examples):.1f}%)")
print(f"  Test:  {len(test_data)} ({100*len(test_data)/len(tokenized_examples):.1f}%)")

[SPLIT]
  Train: 429 (70.0%)
  Val:   92 (15.0%)
  Test:  92 (15.0%)


In [25]:
# Convertir en HuggingFace Datasets
train_dataset = Dataset.from_dict({
    'input_ids': [ex['input_ids'] for ex in train_data],
    'attention_mask': [ex['attention_mask'] for ex in train_data],
    'labels': [ex['labels'] for ex in train_data],
})

val_dataset = Dataset.from_dict({
    'input_ids': [ex['input_ids'] for ex in val_data],
    'attention_mask': [ex['attention_mask'] for ex in val_data],
    'labels': [ex['labels'] for ex in val_data],
})

test_dataset = Dataset.from_dict({
    'input_ids': [ex['input_ids'] for ex in test_data],
    'attention_mask': [ex['attention_mask'] for ex in test_data],
    'labels': [ex['labels'] for ex in test_data],
})

dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

print(f"[OK] HuggingFace Datasets created")
print(f"  Train shape: {train_dataset.shape}")
print(f"  Val shape: {val_dataset.shape}")
print(f"  Test shape: {test_dataset.shape}")

[OK] HuggingFace Datasets created
  Train shape: (429, 3)
  Val shape: (92, 3)
  Test shape: (92, 3)


---
## Step 3 : Fine-tuning CAMeL-BERT

In [26]:
# Charger le modèle pré-entraîné
print(f"[INFO] Loading model: {model_name}")

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=2,  # Binary classification: boundary (1) vs non-boundary (0)
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2,
)

print(f"[OK] Model loaded")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

[INFO] Loading model: CAMeL-Lab/bert-base-arabic-camelbert-ca


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-ca
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight             

[OK] Model loaded
  Total parameters: 108,492,290
  Trainable: 108,492,290


In [27]:
# Configuration d'entraînement
output_dir = Path('checkpoints/camelbert_binary_classification')
output_dir.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(output_dir),

    # Hyperparamètres
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=100,

    # Évaluation et logging
    eval_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",

    # Misc
    seed=42,
    dataloader_pin_memory=True,
    optim="adamw_torch",
)

print(f"[OK] Training config set")
print(f"  Output dir: {output_dir}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  LR: {training_args.learning_rate}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")

[OK] Training config set
  Output dir: checkpoints/camelbert_binary_classification
  Epochs: 5
  LR: 2e-05
  Batch size: 8


In [28]:
# Métriques d'évaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)

    # Aplatir les labels (ils sont des listes de 512 labels)
    labels_flat = labels.flatten()
    predictions_flat = predictions.flatten()

    # Ignorer les labels de padding (0) pour une métrique plus propre
    mask = labels_flat != -100
    labels_clean = labels_flat[mask]
    predictions_clean = predictions_flat[mask]

    accuracy = accuracy_score(labels_clean, predictions_clean)
    f1 = f1_score(labels_clean, predictions_clean, average='binary', zero_division=0)

    return {
        'accuracy': accuracy,
        'f1': f1,
    }

print("[OK] Metrics function defined")

[OK] Metrics function defined


In [29]:
# Créer le Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("[OK] Trainer created")

[OK] Trainer created


In [30]:
# Fine-tuning
print("[INFO] Starting fine-tuning...\n")

train_result = trainer.train()

print(f"\n[OK] Fine-tuning completed")
print(f"  Training loss: {train_result.training_loss:.4f}")

[INFO] Starting fine-tuning...



Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.101112,0.023735,0.993631,0.929511
2,0.010771,0.010449,0.996837,0.966144
3,0.005381,0.009093,0.997495,0.973084
4,0.002917,0.009403,0.997070,0.968807
5,0.002923,0.008777,0.997622,0.974476


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


[OK] Fine-tuning completed
  Training loss: 0.0599


---
## Step 4 : Évaluation

In [31]:
# Évaluation sur val set
print("[INFO] Evaluating on validation set...")
val_results = trainer.evaluate(eval_dataset=dataset['validation'])

print(f"\n[VALIDATION RESULTS]")
for key, val in val_results.items():
    print(f"  {key}: {val:.4f}")

[INFO] Evaluating on validation set...



[VALIDATION RESULTS]
  eval_loss: 0.0088
  eval_accuracy: 0.9976
  eval_f1: 0.9745
  eval_runtime: 3.1164
  eval_samples_per_second: 29.5210
  eval_steps_per_second: 3.8510
  epoch: 5.0000


In [34]:
# Évaluation sur test set
print("[INFO] Evaluating on test set...")

# Predictions
predictions = trainer.predict(dataset['test'])
preds = np.argmax(predictions.predictions, axis=-1)  # [batch_size, seq_len]
labels_test = predictions.label_ids  # shape [batch_size, seq_len]

# Aplatir
preds_test = preds.flatten()
labels_test = labels_test.flatten()

# (Optionnel) Ignorer le padding si vous utilisez -100
mask = labels_test != -100
if mask.sum() > 0:
    preds_test = preds_test[mask]
    labels_test = labels_test[mask]


# Métriques
test_accuracy = accuracy_score(labels_test, preds_test)
test_f1 = f1_score(labels_test, preds_test, average='binary', zero_division=0)
test_cm = confusion_matrix(labels_test, preds_test)

print(f"\n[TEST RESULTS]")
print(f"  Accuracy: {test_accuracy:.4f}")
print(f"  F1 Score: {test_f1:.4f}")
print(f"\n[CONFUSION MATRIX]")
print(f"  TN FP")
print(f"  {test_cm[0][0]} {test_cm[0][1]}")
print(f"  FN TP")
print(f"  {test_cm[1][0]} {test_cm[1][1]}")
print(f"\n[CLASSIFICATION REPORT]")
print(classification_report(labels_test, preds_test, target_names=['non-boundary', 'boundary']))

[INFO] Evaluating on test set...



[TEST RESULTS]
  Accuracy: 0.9979
  F1 Score: 0.9811

[CONFUSION MATRIX]
  TN FP
  44379 89
  FN TP
  12 2624

[CLASSIFICATION REPORT]
              precision    recall  f1-score   support

non-boundary       1.00      1.00      1.00     44468
    boundary       0.97      1.00      0.98      2636

    accuracy                           1.00     47104
   macro avg       0.98      1.00      0.99     47104
weighted avg       1.00      1.00      1.00     47104



---
## Step 5 : Sauvegarde sur Google Drive

In [35]:
# Sauvegarder le modèle
model_output_dir = Path('checkpoints/camelbert_binary_classification_final')
model_output_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(model_output_dir))
tokenizer.save_pretrained(str(model_output_dir))

print(f"[OK] Model saved to {model_output_dir}")
print(f"  Files: {list(model_output_dir.glob('*'))}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[OK] Model saved to checkpoints/camelbert_binary_classification_final
  Files: [PosixPath('checkpoints/camelbert_binary_classification_final/config.json'), PosixPath('checkpoints/camelbert_binary_classification_final/model.safetensors'), PosixPath('checkpoints/camelbert_binary_classification_final/training_args.bin'), PosixPath('checkpoints/camelbert_binary_classification_final/tokenizer_config.json'), PosixPath('checkpoints/camelbert_binary_classification_final/tokenizer.json')]


In [36]:
# Sauvegarder les résultats
results_dir = Path('results/binary_classification_finetuning')
results_dir.mkdir(parents=True, exist_ok=True)

# Résumé
summary = {
    'model': model_name,
    'approach': 'binary_classification_per_segment',
    'dataset': {
        'total_examples': len(tokenized_examples),
        'train_size': len(train_data),
        'val_size': len(val_data),
        'test_size': len(test_data),
    },
    'training': {
        'epochs': training_args.num_train_epochs,
        'learning_rate': training_args.learning_rate,
        'batch_size': training_args.per_device_train_batch_size,
        'training_loss': float(train_result.training_loss),
    },
    'validation_results': val_results,
    'test_results': {
        'accuracy': float(test_accuracy),
        'f1': float(test_f1),
        'confusion_matrix': test_cm.tolist(),
    }
}

with open(results_dir / 'finetuning_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"[OK] Results saved to {results_dir}")

[OK] Results saved to results/binary_classification_finetuning


In [37]:
# Sauvegarder les prédictions détaillées
detailed_results = []
for i, (test_idx, pred, label) in enumerate(zip(test_indices, preds_test[:len(test_indices)], labels_test[:len(test_indices)])):
    detailed_results.append({
        'index': int(test_idx),
        'prediction': int(pred),
        'label': int(label),
        'correct': bool(pred == label),
    })

with open(results_dir / 'detailed_predictions.jsonl', 'w', encoding='utf-8') as f:
    for res in detailed_results:
        f.write(json.dumps(res) + '\n')

print(f"[OK] Detailed predictions saved")

[OK] Detailed predictions saved


---
## 📊 Résumé Final

In [38]:
print(f"""
╔════════════════════════════════════════════════════════════╗
║     CAMeL-BERT Binary Classification Fine-tuning DONE     ║
╚════════════════════════════════════════════════════════════╝

📊 DATASET
  • Total examples: {len(tokenized_examples)}
  • Train/Val/Test: {len(train_data)}/{len(val_data)}/{len(test_data)}
  • Boundary tokens ratio: {label_stats[1] / sum(label_stats.values()):.1%}

🤖 MODEL
  • Base: {model_name}
  • Task: Binary classification (boundary vs non-boundary)
  • Parameters: {sum(p.numel() for p in model.parameters()):,}

📈 RESULTS
  • Validation Accuracy: {val_results.get('eval_accuracy', 0):.4f}
  • Validation F1: {val_results.get('eval_f1', 0):.4f}
  • Test Accuracy: {test_accuracy:.4f}
  • Test F1: {test_f1:.4f}

💾 OUTPUTS (on Google Drive)
  • Model: checkpoints/camelbert_binary_classification_final/
  • Results: results/binary_classification_finetuning/
    - finetuning_summary.json
    - detailed_predictions.jsonl

✅ Next steps:
  1. Review results/binary_classification_finetuning/finetuning_summary.json
  2. Evaluate model on real test corpus
  3. If satisfactory, proceed to Piste 2 (Span-based) or deploy
""")


╔════════════════════════════════════════════════════════════╗
║     CAMeL-BERT Binary Classification Fine-tuning DONE     ║
╚════════════════════════════════════════════════════════════╝

📊 DATASET
  • Total examples: 613
  • Train/Val/Test: 429/92/92
  • Boundary tokens ratio: 5.1%

🤖 MODEL
  • Base: CAMeL-Lab/bert-base-arabic-camelbert-ca
  • Task: Binary classification (boundary vs non-boundary)
  • Parameters: 108,492,290

📈 RESULTS
  • Validation Accuracy: 0.9976
  • Validation F1: 0.9745
  • Test Accuracy: 0.9979
  • Test F1: 0.9811

💾 OUTPUTS (on Google Drive)
  • Model: checkpoints/camelbert_binary_classification_final/
  • Results: results/binary_classification_finetuning/
    - finetuning_summary.json
    - detailed_predictions.jsonl

✅ Next steps:
  1. Review results/binary_classification_finetuning/finetuning_summary.json
  2. Evaluate model on real test corpus
  3. If satisfactory, proceed to Piste 2 (Span-based) or deploy

